In [119]:
import pandas as pd
import numpy as np
import pickle
from umap import UMAP
import plotly.express as px
import plotly.graph_objects as go

# 1. Get pairwise differences between Standard, S6+Standard, Standard+S2_prime
standard = S5, S4, S3, S2, S1, S1_prime

In [ ]:
with open("/Users/hannah/code/rotation1/rotation1_code/Mpro_analysis/Fragalysis_SARS_Mpro_embeddings_1.pkl", "rb") as f:
    em1 = pickle.load(f)
with open("/Users/hannah/code/rotation1/rotation1_code/Mpro_analysis/Fragalysis_SARS_Mpro_embeddings_2.pkl", "rb") as f:
    em2 = pickle.load(f)
with open("/Users/hannah/code/rotation1/rotation1_code/Mpro_analysis/Fragalysis_SARS_Mpro_embeddings_3.pkl", "rb") as f:
    em3 = pickle.load(f)

graph_em1 = np.array([e['graph_embedding'] for e in em1])
graph_em2 = np.array([e['graph_embedding'] for e in em2])
graph_em3 = np.array([e['graph_embedding'] for e in em3])

mean_graph_em = np.mean(np.stack([graph_em1, graph_em2, graph_em3]), axis=0)

In [58]:
ids = ([e["id"] for e in em1])

graph_em_with_name = pd.DataFrame({
    "pdb_id": ids,
    "mean_graph_em": list(mean_graph_em)
})

In [ ]:
df = pd.read_csv("/Users/hannah/code/rotation1/rotation1_code/Mpro_analysis/fragalysis_parsed_full_1319.csv")

In [80]:
categories = {
    "standard":           (["S5","S4","S3","S2","S1","S1_prime"], ["S6", "S2_prime", "S3_prime", "S4_prime"]),
    "standard_S2_prime":  (["S5","S4","S3","S2","S1","S1_prime","S2_prime"], ["S6"]),
    "S6_standard":        (["S6","S5","S4","S3","S2","S1","S1_prime"], ["S2_prime"]),
    "S6_standard_S2_prime":(["S6","S5","S4","S3","S2","S1","S1_prime","S2_prime"], []),
}

results = {
    name: df.loc[
        df["subsites"].apply(lambda c: all(s in c for s in must) and not any(s in c for s in exclude)),
        "pdb_id"
    ].tolist()
    for name, (must, exclude) in categories.items()
}

In [81]:
for category, pdb_ids in results.items():
    print(category, len(pdb_ids))

standard 754
standard_S2_prime 308
S6_standard 54
S6_standard_S2_prime 13


In [83]:
graph_em_with_name["pdb_id"] = graph_em_with_name["pdb_id"].str.split(".").str[0]
graph_em_indexed = graph_em_with_name.set_index("pdb_id")

In [84]:
#assign the embeddings to each category based on their pdb
category_em = {
    "standard": [],
    "standard_S2_prime": [],
    "S6_standard": [],
    "S6_standard_S2_prime": []
}

for category, pdb_ids in results.items():
    for pdb_id in pdb_ids:
        stripped = pdb_id.split(".")[0]
        if stripped in graph_em_indexed.index:
            em = graph_em_indexed.loc[stripped, "mean_graph_em"]
            category_em[category].append(em)

In [86]:
for category, ems in category_em.items():
    print(category, len(ems))

print(category_em)

standard 754
standard_S2_prime 308
S6_standard 54
S6_standard_S2_prime 13
{'standard': [array([ 0.13048904,  0.07555384, -0.00945912, -0.00226592,  0.04301573,
       -0.14447045,  0.13599025, -0.01726239, -0.2664738 , -0.02572932,
        0.05774921,  0.12456775, -0.13038807, -0.12724096,  0.1368504 ,
       -0.1050033 ,  0.09004098, -0.0937951 , -0.06286192, -0.04176337,
       -0.20945364, -0.03116175, -0.2380838 ,  0.23265189,  0.04380317,
       -0.04445185,  0.06563818, -0.07038126, -0.17219587, -0.2546656 ,
        0.02567251,  0.10172117], dtype=float32), array([ 0.12744655,  0.04740285, -0.00804991, -0.00670518,  0.03787939,
       -0.15760934,  0.13401753, -0.03148462, -0.27620173, -0.00917372,
        0.05459073,  0.12626745, -0.12663677, -0.11547759,  0.10921638,
       -0.10987439,  0.08831184, -0.09453845, -0.07426226, -0.04901301,
       -0.18044978, -0.02476582, -0.24372305,  0.21505417,  0.05284422,
       -0.04891981,  0.10115509, -0.10002661, -0.18547599, -0.26778296

In [97]:
x = np.vstack([category_em["standard"].copy(), category_em["standard_S2_prime"].copy()])
y = np.concatenate([
    np.zeros(len(category_em["standard"])),
    np.ones(len(category_em["standard_S2_prime"]))
])

In [98]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

clf = LogisticRegression()
clf.fit(X_train, y_train)
print(clf.score(X_test, y_test))

0.7136150234741784


In [121]:
direction = clf.coef_[0]
direction_normalised = direction / np.linalg.norm(direction)


In [123]:
mean_standard = np.mean(category_em["standard"], axis=0)
arrow_end = mean_standard + direction_normalised * 0.5  # scale e.g. 0.5 or 1

## 1.1 getting it onto UMAP

In [102]:
reducer = UMAP(random_state=42, n_neighbors=30)
umap_embedding = reducer.fit_transform(mean_graph_em)

/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [105]:
plot_df = pd.DataFrame({
    "x": umap_embedding[:, 0],
    "y": umap_embedding[:, 1],
    "pdb_id": ids,
    "subsites": df["subsites"].astype(str),
    "smiles": df["lig_smiles"]
})

def classify_binder(subsites):
    s = str(subsites)
    if "others" in s:
        return "Non active site"
    elif "S3_prime" in s or "S4_prime" in s:
        return "S3_prime/S4_prime (T24 contact)"
    elif "S6" in s and "S2_prime" in s:
        return "S6 + Standard + S2_prime"
    elif "S6" in s:
        return "S6 + Standard"
    elif "S2_prime" in s:
        return "Standard + S2_prime"
    elif "S5" in s and "S1_prime" in s:
        return "Standard (S5 - S1_prime)"
    elif "S4" in s and "S1_prime" in s:
        return "S4 - S1_prime"
    else:
        return "Minor combinations"

plot_df["binding_class"] = plot_df["subsites"].apply(classify_binder)

fig = px.scatter(
    plot_df,
    x="x", y="y",
    color="binding_class",
    hover_data=["pdb_id", "subsites", "smiles"],
    title="Fragalysis SARS-CoV-2 Mpro UMAP - coloured by subsites",
    color_discrete_map={
        "Standard (S5 - S1_prime)": "lightgrey",
        "Standard + S2_prime": "royalblue",
        "S4 - S1_prime": "pink",
        "S6 + Standard": "tomato",
        "S6 + Standard + S2_prime": "purple",
        "S3_prime/S4_prime (T24 contact)": "gold",
        "Minor combinations": "orange",
        "Non active site": "green"
    },
    width = 800, height = 500
)

fig.update_traces(marker=dict(size=5, opacity=0.9))
fig.update_layout(yaxis=dict(scaleanchor="x", scaleratio=1))
fig.show()

In [117]:
start_umap = reducer.transform([mean_standard])
end_umap = reducer.transform([arrow_end])

In [109]:
print(start_umap, end_umap)

[[ 7.47609 -2.89024]] [[ 0.25569138 -2.1017694 ]]


In [110]:
fig.add_annotation(
    x=end_umap[0][0],
    y=end_umap[0][1],
    ax=start_umap[0][0],
    ay=start_umap[0][1],
    xref="x", yref="y",
    axref="x", ayref="y",
    showarrow=True,
    arrowhead=3,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="black",
    text="S2_prime direction"
)

## 1.2 getting into 3D umap 

In [112]:
reducer_3d = UMAP(random_state=42, n_neighbors=30, n_components=3)
umap_embedding_3d = reducer_3d.fit_transform(mean_graph_em)

/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [114]:
plot_df_3d = pd.DataFrame({
    "x": umap_embedding_3d[:, 0],
    "y": umap_embedding_3d[:, 1],
    "z": umap_embedding_3d[:, 2],
    "pdb_id": ids,
    "subsites": df["subsites"].astype(str),
    "smiles": df["lig_smiles"]
})

plot_df_3d["binding_class"] = plot_df["subsites"].apply(classify_binder)

fig_3d = px.scatter_3d(
    plot_df_3d,
    x="x", y="y", z="z",
    color="binding_class",
    hover_data=["pdb_id", "subsites", "smiles"],
    title="Fragalysis SARS-CoV-2 Mpro UMAP - coloured by subsites",
    color_discrete_map={
        "Standard (S5 - S1_prime)": "lightgrey",
        "Standard + S2_prime": "royalblue",
        "S4 - S1_prime": "pink",
        "S6 + Standard": "tomato",
        "S6 + Standard + S2_prime": "purple",
        "S3_prime/S4_prime (T24 contact)": "gold",
        "Minor combinations": "orange",
        "Non active site": "green"
    },
    width = 800, height = 500
)

fig_3d.update_traces(marker=dict(size=3, opacity=0.95))
fig_3d.update_layout(yaxis=dict(scaleanchor="x", scaleratio=1))
fig_3d.show()

In [124]:
start_umap_3d = reducer_3d.transform([mean_standard])
end_umap_3d = reducer_3d.transform([arrow_end])
print(start_umap_3d, end_umap_3d)

[[ 6.1717443 -2.1501803  6.4451966]] [[ 2.4475303   0.96291316 10.610232  ]]


In [125]:
# line
fig_3d.add_trace(go.Scatter3d(
    x=[start_umap_3d[0][0], end_umap_3d[0][0]],
    y=[start_umap_3d[0][1], end_umap_3d[0][1]],
    z=[start_umap_3d[0][2], end_umap_3d[0][2]],
    mode="lines",
    line=dict(color="black", width=5),
    name="S2_prime direction"
))

# arrowhead
fig_3d.add_trace(go.Cone(
    x=[end_umap_3d[0][0]],
    y=[end_umap_3d[0][1]],
    z=[end_umap_3d[0][2]],
    u=[end_umap_3d[0][0] - start_umap_3d[0][0]],
    v=[end_umap_3d[0][1] - start_umap_3d[0][1]],
    w=[end_umap_3d[0][2] - start_umap_3d[0][2]],
    showscale=False,
    colorscale=[[0, "black"], [1, "black"]]
))